<a href="https://colab.research.google.com/github/Navabhargav/Insurance-Fraud/blob/main/Simple_fraud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

# ----------------------------
# 1) Toy STRUCTURED dataset
# ----------------------------
structured_rows = [
    # claim_amount, policy_age_days, num_prev_claims, days_since_last_claim, provider_id, region, claim_type, payment_method, label
    (120000, 12, 3, 40, "P88", "South", "accident", "cash", 1),
    (5000,   800, 0, 999, "P10", "North", "theft", "card", 0),
    (30000,  60,  2, 70,  "P88", "South", "accident", "cash", 1),
    (8000,   500, 1, 200, "P20", "East",  "accident", "card", 0),
    (150000, 20,  4, 30,  "P88", "South", "accident", "cash", 1),
    (7000,   900, 0, 400, "P30", "West",  "theft", "card", 0),
]

import pandas as pd
df = pd.DataFrame(structured_rows, columns=[
    "claim_amount","policy_age_days","num_prev_claims","days_since_last_claim",
    "provider_id","customer_region","claim_type","payment_method","label_fraud"
])

X_struct = df.drop(columns=["label_fraud"])
y = df["label_fraud"]

num_cols = ["claim_amount","policy_age_days","num_prev_claims","days_since_last_claim"]
cat_cols = ["provider_id","customer_region","claim_type","payment_method"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

structured_model = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

X_train, X_test, y_train, y_test = train_test_split(X_struct, y, test_size=0.33, random_state=42)
structured_model.fit(X_train, y_train)

# Example new claim (structured input)
new_claim_struct = pd.DataFrame([{
    "claim_amount": 140000,
    "policy_age_days": 15,
    "num_prev_claims": 3,
    "days_since_last_claim": 45,
    "provider_id": "P88",
    "customer_region": "South",
    "claim_type": "accident",
    "payment_method": "cash"
}])

risk_score_struct = structured_model.predict_proba(new_claim_struct)[0,1]

# ----------------------------
# 2) Toy TEXT dataset + model
# ----------------------------
texts = [
    "minor accident but invoice revised multiple times and amount increased",
    "customer reports theft, police report attached, consistent timeline",
    "provider changed invoice amounts three times, suspicious billing pattern",
    "accident description consistent with photos and repair estimate",
    "staged collision suspected, inflated medical bill, repeated edits",
    "clear incident report and stable invoice, no suspicious pattern"
]
labels = np.array([1,0,1,0,1,0])

text_model = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=1)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

text_model.fit(texts, labels)

new_claim_text = "minor accident, but the invoice was revised three times and amounts increased"
risk_score_text = text_model.predict_proba([new_claim_text])[0,1]

# ----------------------------
# 3) Combine scores (simple weighted ensemble)
# ----------------------------
final_risk_score = 0.6 * risk_score_struct + 0.4 * risk_score_text

# Decision thresholds
if final_risk_score < 0.40:
    decision = "auto_approve"
elif final_risk_score < 0.65:
    decision = "manual_review"
else:
    decision = "escalate"

print("risk_score_struct:", round(float(risk_score_struct), 3))
print("risk_score_text:", round(float(risk_score_text), 3))
print("final_risk_score:", round(float(final_risk_score), 3))
print("decision:", decision)

# ----------------------------
# 4) RAG prompt ONLY if escalate/manual_review band (example: escalate)
# ----------------------------
retrieved_evidence = [
    {"chunk_id": "CH1", "text": "Provider P88 has multiple historical confirmed fraud cases involving inflated invoices."},
    {"chunk_id": "CH7", "text": "Policy excludes coverage if fraud or intentional misrepresentation is confirmed."},
    {"chunk_id": "CH9", "text": "Similar past case: staged collision pattern + repeated invoice revisions before submission."}
]

def build_prompt(claim_struct, claim_text, risk_s, risk_t, final_score, evidence_chunks):
    claim_summary = {
        "claim_amount": int(claim_struct["claim_amount"].iloc[0]),
        "policy_age_days": int(claim_struct["policy_age_days"].iloc[0]),
        "num_prev_claims": int(claim_struct["num_prev_claims"].iloc[0]),
        "days_since_last_claim": int(claim_struct["days_since_last_claim"].iloc[0]),
        "provider_id": claim_struct["provider_id"].iloc[0],
        "customer_region": claim_struct["customer_region"].iloc[0],
        "claim_type": claim_struct["claim_type"].iloc[0],
        "payment_method": claim_struct["payment_method"].iloc[0],
    }

    evidence_text = "\n".join([f"[chunk_id={e['chunk_id']}] {e['text']}" for e in evidence_chunks])

    prompt = f"""
SYSTEM:
You are a fraud investigation assistant.
Rules:
1) Use ONLY the Evidence section to justify reasons. If evidence is insufficient, say "insufficient_evidence".
2) Cite evidence by chunk_id and include short direct quotes.
3) Output MUST be valid JSON ONLY. No extra text.

USER:
Claim Summary (structured):
{claim_summary}

Claim Text:
{claim_text}

Model Signals:
- risk_score_struct: {risk_s:.3f}
- risk_score_text: {risk_t:.3f}
- final_risk_score: {final_score:.3f}

Evidence (retrieved chunks):
{evidence_text}

Return JSON with this schema:
{{
  "fraud_label": "fraud|not_fraud|review",
  "confidence": 0.0,
  "top_reasons": ["..."],
  "evidence_chunks": [{{"chunk_id":"...", "quote":"..."}}],
  "recommended_action": "auto_approve|manual_review|escalate",
  "missing_info": ["..."]
}}
""".strip()
    return prompt

if decision in ["manual_review", "escalate"]:
    prompt = build_prompt(new_claim_struct, new_claim_text, risk_score_struct, risk_score_text, final_risk_score, retrieved_evidence)
    print("\n--- PROMPT SENT TO LLM ---\n")
    print(prompt[:900], "...\n")

    # Simulated LLM JSON output (what you'd expect)
    llm_json_output = {
        "fraud_label": "review",
        "confidence": round(float(final_risk_score), 2),
        "top_reasons": [
            "Provider appears in historical confirmed fraud patterns involving inflated invoices",
            "Invoice revision pattern matches prior staged collision case notes"
        ],
        "evidence_chunks": [
            {"chunk_id": "CH1", "quote": "Provider P88 has multiple historical confirmed fraud cases involving inflated invoices."},
            {"chunk_id": "CH9", "quote": "Similar past case: staged collision pattern + repeated invoice revisions before submission."}
        ],
        "recommended_action": "escalate",
        "missing_info": ["Police report verification", "Original invoice before revisions"]
    }

    print("--- SIMULATED LLM JSON OUTPUT ---\n")
    print(llm_json_output)

risk_score_struct: 1.0
risk_score_text: 0.58
final_risk_score: 0.832
decision: escalate

--- PROMPT SENT TO LLM ---

SYSTEM:
You are a fraud investigation assistant.
Rules:
1) Use ONLY the Evidence section to justify reasons. If evidence is insufficient, say "insufficient_evidence".
2) Cite evidence by chunk_id and include short direct quotes.
3) Output MUST be valid JSON ONLY. No extra text.

USER:
Claim Summary (structured):
{'claim_amount': 140000, 'policy_age_days': 15, 'num_prev_claims': 3, 'days_since_last_claim': 45, 'provider_id': 'P88', 'customer_region': 'South', 'claim_type': 'accident', 'payment_method': 'cash'}

Claim Text:
minor accident, but the invoice was revised three times and amounts increased

Model Signals:
- risk_score_struct: 1.000
- risk_score_text: 0.580
- final_risk_score: 0.832

Evidence (retrieved chunks):
[chunk_id=CH1] Provider P88 has multiple historical confirmed fraud cases involving inflated invoices.
[chunk_id=CH7] Policy excludes coverage if fraud o